In [5]:
%pip install numpy pandas duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.9 MB/s  0:00:006m0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb

conn = duckdb.connect("main.db")

In [10]:
# Which pollutants appear most frequently as "prominent pollutants"?

conn.execute(""" 
WITH pollutants_unnested AS (
SELECT
    date,
    state,
    area,
    air_quality_status,
    aqi_value,
    UNNEST(STRING_SPLIT(prominent_pollutants, ',')) AS pollutants        
FROM air_quality
)

/*
SELECT
    pollutants,
    AVG(aqi_value) AS avg_aqi,
    COUNT(DISTINCT date) AS pollutant_occurence
FROM pollutants_unnested
GROUP BY pollutants
ORDER BY pollutant_occurence DESC
*/
             
SELECT
    pollutants,
    COUNT(DISTINCT date) AS total_occurrences,
    AVG(aqi_value) AS avg_aqi_when_present,
    COUNT(DISTINCT area) AS cities_affected
FROM pollutants_unnested
WHERE air_quality_status IN ('Poor', 'Very Poor', 'Severe')
GROUP BY pollutants
ORDER BY total_occurrences DESC
""").fetch_df()

,pollutants,total_occurrences,avg_aqi_when_present,cities_affected
0,PM2.5,2938,278.565462,261
1,PM10,2660,267.305885,210
2,O3,1532,244.716811,184
3,NO2,309,254.996904,56
4,CO,18,289.277778,13
5,SO2,13,250.153846,5


In [27]:
# Do multiple pollutants appear together?
conn.execute("""

WITH pollutant_combos AS (
    SELECT
        date,
        area,
        prominent_pollutants,
        LENGTH(prominent_pollutants) - LENGTH(REPLACE(prominent_pollutants, ',', '')) + 1 AS num_pollutants
    FROM air_quality
    WHERE air_quality_status IN ('Poor', 'Very Poor', 'Severe')
)

SELECT
    prominent_pollutants,
    COUNT(*) AS occurrences,
    AVG(num_pollutants) AS avg_pollutants_together
FROM pollutant_combos
GROUP BY prominent_pollutants
ORDER BY occurrences DESC
LIMIT 20
""").fetch_df()

,prominent_pollutants,occurrences,avg_pollutants_together
0,PM2.5,45091,1.0
1,PM10,10258,1.0
2,"PM2.5,PM10",3979,2.0
3,O3,1808,1.0
4,"PM2.5,O3",425,2.0
5,NO2,214,1.0
6,"PM10,O3",168,2.0
7,"O3,PM2.5,PM10",126,3.0
8,"PM2.5,NO2",62,2.0
9,"PM10,NO2",25,2.0


*PM2.5 is the most commonly occurring pollutant and since it is also one of the deadliest pollutants - the company needs to prioritise PM2.5 targeting*

In [30]:
# Which cities consistently show Poor/Very Poor/Severe AQI?

conn.execute(""" 

WITH city_stats AS (
    SELECT
        area,
        state,
        COUNT(DISTINCT date) AS total_days_monitored,
        COUNT(DISTINCT CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN date END) AS bad_air_days,
        AVG(CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN aqi_value END) AS avg_bad_day_aqi,
        MAX(aqi_value) AS worst_aqi_ever
    FROM air_quality
    GROUP BY area, state
),

bad_air_days_pct AS (
SELECT
    area,
    state,
    total_days_monitored,
    bad_air_days,
    ROUND(100.0 * bad_air_days / total_days_monitored, 2) AS pct_bad_days,
    avg_bad_day_aqi,
    worst_aqi_ever
FROM city_stats
WHERE bad_air_days > 100  -- Filter for cities with sustained issues
ORDER BY pct_bad_days DESC, bad_air_days DESC
)
             
SELECT
    CASE WHEN pct_bad_days > 60.0 THEN 'Crisis'
    WHEN pct_bad_days BETWEEN 40.0 AND 60.0 THEN 'High Risk'
    WHEN pct_bad_days BETWEEN 20.0 AND 39.9 THEN 'Mid Risk'
    ELSE 'Low Risk'
    END AS risk_tier,
    COUNT(area) AS cities
FROM bad_air_days_pct
GROUP BY 1
""").fetch_df()

,risk_tier,cities
0,Crisis,1
1,High Risk,16
2,Mid Risk,60
3,Low Risk,62


*Cities with more than 50% bad days => desperate need => strong market fit*

In [16]:
# What's the trend over time - improving or degrading?

conn.execute(""" 
WITH yearly_avg AS (
    SELECT
        area,
        EXTRACT('year' FROM date) AS year,
        AVG(aqi_value) AS avg_aqi,
        COUNT(DISTINCT CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN date END) AS bad_days,
        COUNT(DISTINCT date) AS total_days
    FROM air_quality
    GROUP BY area, EXTRACT('year' FROM date)
),
trend_calc AS (
    SELECT
        area,
        year,
        avg_aqi,
        bad_days,
        total_days,
        LAG(avg_aqi) OVER (PARTITION BY area ORDER BY year) AS prev_year_aqi,
        LAG(bad_days) OVER (PARTITION BY area ORDER BY year) AS prev_year_bad_days
    FROM yearly_avg
)

SELECT
    area,
    year,
    avg_aqi,
    prev_year_aqi,
    ROUND(avg_aqi - prev_year_aqi, 2) AS aqi_change,
    bad_days,
    prev_year_bad_days,
    bad_days - prev_year_bad_days AS bad_days_change,
    CASE 
        WHEN avg_aqi - prev_year_aqi > 10 THEN 'Worsening'
        WHEN avg_aqi - prev_year_aqi < -10 THEN 'Improving'
        ELSE 'Stable'
    END AS trend_status
FROM trend_calc
ORDER BY area, year

""").fetch_df()

,area,year,avg_aqi,prev_year_aqi,aqi_change,bad_days,prev_year_bad_days,bad_days_change,trend_status
0,Agartala,2020,153.450980,NaN,NaN,8,<NA>,<NA>,Stable
1,Agartala,2021,102.346505,153.450980,-51.10,44,8,36,Improving
2,Agartala,2022,111.895028,102.346505,9.55,86,44,42,Stable
3,Agartala,2023,155.321637,111.895028,43.43,108,86,22,Worsening
4,Agartala,2024,139.898413,155.321637,-15.42,94,108,-14,Improving
...,...,...,...,...,...,...,...,...,...
1626,Yamunanagar,2020,154.491124,174.106628,-19.62,95,114,-19,Improving
1627,Yamunanagar,2021,176.817647,154.491124,22.33,122,95,27,Worsening
1628,Yamunanagar,2022,165.410828,176.817647,-11.41,100,122,-22,Improving
1629,Yamunanagar,2023,132.850794,165.410828,-32.56,41,100,-59,Improving


In [ ]:
# Cities with worsening trends
conn.execute("""
WITH yearly_avg AS (
    SELECT
        area,
        EXTRACT('year' FROM date) AS year,
        AVG(aqi_value) AS avg_aqi
    FROM air_quality
    GROUP BY area, EXTRACT('year' FROM date)
),

year_bounds AS (
    SELECT
        area,
        MIN(year) AS first_year,
        MAX(year) AS last_year
    FROM yearly_avg
    GROUP BY area
)

SELECT
    yb.area,
    yb.first_year,
    yb.last_year,
    y1.avg_aqi AS early_aqi,
    y2.avg_aqi AS recent_aqi,
    y2.avg_aqi - y1.avg_aqi AS aqi_change
FROM year_bounds yb
JOIN yearly_avg y1 
    ON yb.area = y1.area AND yb.first_year = y1.year
JOIN yearly_avg y2 
    ON yb.area = y2.area AND yb.last_year = y2.year
WHERE y2.avg_aqi > y1.avg_aqi + 20   -- significant worsening
ORDER BY aqi_change DESC
""").fetch_df()

,area,first_year,last_year,early_aqi,recent_aqi,aqi_change
0,Darbhanga,2021,2023,248.863636,364.714286,115.850649
1,Hajipur,2020,2025,98.784553,196.636364,97.851811
2,Panchkula,2015,2025,92.000000,185.416667,93.416667
3,Gurgaon,2015,2018,146.333333,214.957198,68.623865
4,Tumakuru,2023,2025,95.818182,151.854545,56.036364
5,Gangtok,2022,2025,33.594444,82.962963,49.368519
6,Kunjemura,2023,2025,89.469613,137.254777,47.785164
7,Milupara,2023,2025,60.752874,105.792079,45.039206
8,Shillong,2019,2025,35.615385,75.927928,40.312543
9,Tumidih,2023,2025,81.409756,120.717172,39.307416


In [31]:
# Combine AQI + Population for market potential

conn.execute("""
WITH city_pollution AS (
    SELECT
        area,
        state,
        AVG(aqi_value) AS avg_aqi,
        COUNT(DISTINCT CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN date END) * 100.0 / COUNT(DISTINCT date) AS pct_bad_days,
        COUNT(DISTINCT date) AS total_days
    FROM air_quality
    GROUP BY area, state
),
state_population AS (
    SELECT
        state,
        AVG(CASE WHEN gender = 'Total' THEN value * 1000 END) AS population
    FROM population_data
    WHERE year = 2024  -- or most recent year available
    GROUP BY state
)

SELECT
    c.area,
    c.state,
    c.avg_aqi,
    c.pct_bad_days,
    c.total_days,
    p.population,
    CASE 
        WHEN c.pct_bad_days > 60 THEN 'Tier 1 Priority'
        WHEN c.pct_bad_days > 40 THEN 'Tier 2 Priority'
        ELSE 'Lower Priority'
    END AS market_tier
FROM city_pollution c
LEFT JOIN state_population p ON c.state = p.state
WHERE c.total_days > 1000  -- At least ~3 years of data
AND p.population > 2000000  -- At least 2M people (market size threshold)
ORDER BY c.pct_bad_days DESC
LIMIT 20
""").fetch_df()

,area,state,avg_aqi,pct_bad_days,total_days,population,market_tier
0,Delhi,Delhi,215.895080,51.372656,3679,2.180300e+07,Tier 2 Priority
1,Bhiwadi,Rajasthan,206.061728,49.421296,2592,2.201900e+07,Tier 2 Priority
2,Ghaziabad,Uttar Pradesh,213.321598,49.322952,2954,5.782467e+07,Tier 2 Priority
3,Greater Noida,Uttar Pradesh,206.425068,47.216816,2569,5.782467e+07,Tier 2 Priority
4,Faridabad,Haryana,200.388872,44.894939,3379,1.315067e+07,Tier 2 Priority
5,NOIDA,Uttar Pradesh,199.969409,43.881713,2942,5.782467e+07,Tier 2 Priority
6,Gurugram,Haryana,188.784468,42.400332,2408,1.315067e+07,Tier 2 Priority
7,Patna,Bihar,186.660480,41.856000,3125,1.599267e+07,Tier 2 Priority
8,Muzaffarpur,Bihar,184.824534,41.764148,3163,1.599267e+07,Tier 2 Priority
9,Chhapra,Bihar,186.193727,40.498155,1084,1.599267e+07,Tier 2 Priority


In [11]:
# Seasonal patterns in AQI

conn.execute(""" 

WITH seasonal_data AS (
    SELECT
        area,
        state,
        date,
        EXTRACT('year' FROM date) AS year,
        EXTRACT('month' FROM date) AS month,
        CASE WHEN EXTRACT('month' FROM date) IN (11,12,1,2) THEN 'Winter'
             WHEN EXTRACT('month' FROM date) IN (3,4,5,6) THEN 'Summer'
             WHEN EXTRACT('month' FROM date) IN (7,8,9,10) THEN 'Monsoon'
        END AS season,
        aqi_value,
        air_quality_status
    FROM air_quality
),

city_level_seasonal_data AS (
    SELECT
        area,
        season,
        AVG(aqi_value),
        COUNT(DISTINCT CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN date END) * 100.0 / COUNT(DISTINCT date) AS pct_bad_days,
        COUNT(DISTINCT date) AS total_days   
    FROM seasonal_data
    GROUP BY area, season
    ORDER BY area
),
             
total_distinct_days AS (
    SELECT
        COUNT(DISTINCT date) AS total_days
    FROM seasonal_data       
)

SELECT
    season,
    AVG(aqi_value) AS avg_aqi,
    COUNT(DISTINCT CASE 
        WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
        THEN date END) AS bad_days,
    MIN(t.total_days) AS total_days,
    COUNT(DISTINCT CASE 
        WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
        THEN date END) * 100.0 /MIN(t.total_days)  AS pct_bad_days
FROM seasonal_data
CROSS JOIN total_distinct_days t
GROUP BY season
ORDER BY avg_aqi DESC
""").fetch_df()

,season,avg_aqi,bad_days,total_days,pct_bad_days
0,Winter,161.267685,1201,3690,32.547425
1,Summer,110.508352,1191,3690,32.276423
2,Monsoon,82.051040,816,3690,22.113821


In [15]:
# Seasonal patterns in AQI

conn.execute(""" 
WITH seasonal_data AS (
    SELECT
        area,
        state,
        date,
        CASE WHEN EXTRACT('month' FROM date) IN (11,12,1,2) THEN 'Winter'
             WHEN EXTRACT('month' FROM date) IN (3,4,5,6) THEN 'Summer'
             WHEN EXTRACT('month' FROM date) IN (7,8,9,10) THEN 'Monsoon'
        END AS season,
        aqi_value,
        air_quality_status
    FROM air_quality
),
city_seasonal_stats AS (
    SELECT
        area,
        state,
        season,
        COUNT(*) AS total_days,
        COUNT(CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN 1 
        END) AS bad_days,
        AVG(aqi_value) AS avg_aqi,
        COUNT(CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN 1 
        END) * 100.0 / COUNT(*) AS pct_bad_days
    FROM seasonal_data
    GROUP BY area, state, season
),
-- Get overall city risk
city_overall_risk AS (
    SELECT
        area,
        state,
        AVG(CASE WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') THEN 1 ELSE 0 END) * 100 AS overall_pct_bad
    FROM air_quality
    GROUP BY area, state
)

-- Now segment by risk tier
SELECT
    CASE 
        WHEN c.overall_pct_bad >= 40 THEN 'High Risk Cities'
        WHEN c.overall_pct_bad >= 20 THEN 'Medium Risk Cities'
        ELSE 'Low Risk Cities'
    END AS city_tier,
    s.season,
    COUNT(DISTINCT s.area) AS num_cities,
    ROUND(AVG(s.avg_aqi), 1) AS avg_aqi,
    ROUND(AVG(s.pct_bad_days), 1) AS avg_pct_bad_days,
    ROUND(MIN(s.pct_bad_days), 1) AS min_pct_bad_days,
    ROUND(MAX(s.pct_bad_days), 1) AS max_pct_bad_days
FROM city_seasonal_stats s
JOIN city_overall_risk c ON s.area = c.area AND s.state = c.state
GROUP BY 
    CASE 
        WHEN c.overall_pct_bad >= 40 THEN 'High Risk Cities'
        WHEN c.overall_pct_bad >= 20 THEN 'Medium Risk Cities'
        ELSE 'Low Risk Cities'
    END,
    s.season
ORDER BY city_tier, 
    CASE s.season 
        WHEN 'Winter' THEN 1 
        WHEN 'Summer' THEN 2 
        WHEN 'Monsoon' THEN 3 
    END
""").fetch_df()

,city_tier,season,num_cities,avg_aqi,avg_pct_bad_days,min_pct_bad_days,max_pct_bad_days
0,High Risk Cities,Winter,18,283.6,81.1,60.5,100.0
1,High Risk Cities,Summer,18,188.7,41.0,23.8,77.6
2,High Risk Cities,Monsoon,19,138.6,21.5,6.2,100.0
3,Low Risk Cities,Winter,213,120.4,12.2,0.0,50.6
4,Low Risk Cities,Summer,217,88.4,2.7,0.0,20.8
5,Low Risk Cities,Monsoon,212,66.7,1.4,0.0,18.4
6,Medium Risk Cities,Winter,60,221.0,55.1,36.1,75.2
7,Medium Risk Cities,Summer,60,140.0,18.0,0.5,31.6
8,Medium Risk Cities,Monsoon,60,98.8,7.8,0.4,18.5


In [22]:
# MASTER DASHBOARD DATASET

conn.execute("""

-- Step 1: Calculate comprehensive AQI metrics by city
WITH city_aqi_metrics AS (
    SELECT
        area,
        state,
        -- Overall metrics
        COUNT(*) AS total_monitoring_days,
        AVG(aqi_value) AS avg_aqi,
        MAX(aqi_value) AS max_aqi_ever,
        STDDEV(aqi_value) AS aqi_volatility,
        
        -- Bad air day metrics
        COUNT(CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN 1 END) AS bad_days,
        COUNT(CASE 
            WHEN air_quality_status IN ('Poor', 'Very Poor', 'Severe') 
            THEN 1 END) * 100.0 / COUNT(*) AS pct_bad_days,
        
        -- Crisis day metrics (Very Poor + Severe only)
        COUNT(CASE 
            WHEN air_quality_status IN ('Very Poor', 'Severe') 
            THEN 1 END) AS crisis_days,
        COUNT(CASE 
            WHEN air_quality_status IN ('Very Poor', 'Severe') 
            THEN 1 END) * 100.0 / COUNT(*) AS pct_crisis_days,
        
        -- Pollutant diversity (more pollutants = more complex problem)
        COUNT(DISTINCT prominent_pollutants) AS unique_pollutant_combinations,
        
        -- Seasonal breakdown
        AVG(CASE 
            WHEN EXTRACT('month' FROM date) IN (11,12,1,2) 
            THEN aqi_value END) AS winter_avg_aqi,
        AVG(CASE 
            WHEN EXTRACT('month' FROM date) IN (3,4,5,6) 
            THEN aqi_value END) AS summer_avg_aqi,
        AVG(CASE 
            WHEN EXTRACT('month' FROM date) IN (7,8,9,10) 
            THEN aqi_value END) AS monsoon_avg_aqi,
            
        -- Winter severity (key selling season)
        COUNT(CASE 
            WHEN EXTRACT('month' FROM date) IN (11,12,1,2)
            AND air_quality_status IN ('Poor', 'Very Poor', 'Severe')
            THEN 1 END) * 100.0 / 
        NULLIF(COUNT(CASE 
            WHEN EXTRACT('month' FROM date) IN (11,12,1,2)
            THEN 1 END), 0) AS winter_pct_bad_days
        
    FROM air_quality
    GROUP BY area, state
),

-- Step 2: Calculate year-over-year trend (FIXED)
city_trends AS (

    -- First compute yearly averages
    WITH yearly_avg AS (
        SELECT
            area,
            state,
            EXTRACT('year' FROM date) AS year,
            AVG(aqi_value) AS avg_aqi
        FROM air_quality
        GROUP BY area, state, EXTRACT('year' FROM date)
    ),

    year_bounds AS (
        SELECT
            area,
            state,
            MIN(year) AS first_year,
            MAX(year) AS last_year
        FROM yearly_avg
        GROUP BY area, state
    )

    SELECT
        yb.area,
        yb.state,
        yb.first_year,
        yb.last_year,
        y1.avg_aqi AS early_aqi,
        y2.avg_aqi AS recent_aqi
    FROM year_bounds yb
    LEFT JOIN yearly_avg y1
        ON yb.area = y1.area
        AND yb.state = y1.state
        AND yb.first_year = y1.year
    LEFT JOIN yearly_avg y2
        ON yb.area = y2.area
        AND yb.state = y2.state
        AND yb.last_year = y2.year
),

-- Step 3: Get state population (latest year available)
state_population AS (
    SELECT
        state,
        AVG(CASE WHEN gender = 'Total' THEN value * 1000 END) AS state_population
    FROM population_data
    WHERE year = (SELECT MAX(year) FROM population_data)
    GROUP BY state
),

-- Step 4: Vehicle registrations as wealth proxy
-- More vehicles per capita = higher income
state_vehicle_metrics AS (
    SELECT
        state,
        -- Total vehicles registered (latest year)
        SUM(CASE WHEN year = (SELECT MAX(year) FROM vehicle_registrations) THEN value END) AS total_vehicles,
        
        -- Electric/Hybrid vehicles as % of total (environmental awareness proxy)
        SUM(CASE 
            WHEN year = (SELECT MAX(year) FROM vehicle_registrations)
            AND fuel IN ('ELECTRIC', 'HYBRID', 'PURE EV')
            THEN value END) * 100.0 / 
        NULLIF(SUM(CASE WHEN year = (SELECT MAX(year) FROM vehicle_registrations) THEN value END), 0) 
            AS pct_ev_vehicles,
        
        -- Motor cars as % (wealth indicator)
        SUM(CASE 
            WHEN year = (SELECT MAX(year) FROM vehicle_registrations)
            AND vehicle_class = 'MOTOR CAR'
            THEN value END) * 100.0 / 
        NULLIF(SUM(CASE WHEN year = (SELECT MAX(year) FROM vehicle_registrations) THEN value END), 0) 
            AS pct_motor_cars
            
    FROM vehicle_registrations
    GROUP BY state
),

-- Step 5: Respiratory disease burden
-- Aggregate respiratory diseases by state
state_disease_burden AS (
    SELECT
        state,
        SUM(cases) AS total_respiratory_cases,
        SUM(deaths) AS total_respiratory_deaths,
        COUNT(DISTINCT district) AS affected_districts,
        
        -- Case fatality rate
        SUM(deaths) * 100.0 / NULLIF(SUM(cases), 0) AS case_fatality_rate
        
    FROM disease_data
    WHERE LOWER(disease_illness_name) LIKE '%respiratory%'
       OR LOWER(disease_illness_name) LIKE '%pneumonia%'
       OR LOWER(disease_illness_name) LIKE '%asthma%'
       OR LOWER(disease_illness_name) LIKE '%bronch%'
       OR LOWER(disease_illness_name) LIKE '%ari%'  -- Acute Respiratory Infection
    GROUP BY state
),

-- Step 6: Combine everything
master_dataset AS (
    SELECT
        -- City identifiers
        a.area AS city,
        a.state,
        
        -- AQI Severity Metrics
        ROUND(a.avg_aqi, 1) AS avg_aqi,
        ROUND(a.max_aqi_ever, 1) AS max_aqi,
        ROUND(a.pct_bad_days, 2) AS pct_bad_days,
        ROUND(a.pct_crisis_days, 2) AS pct_crisis_days,
        ROUND(a.aqi_volatility, 1) AS aqi_volatility,
        
        -- Seasonal patterns
        ROUND(a.winter_avg_aqi, 1) AS winter_avg_aqi,
        ROUND(a.summer_avg_aqi, 1) AS summer_avg_aqi,
        ROUND(a.monsoon_avg_aqi, 1) AS monsoon_avg_aqi,
        ROUND(a.winter_pct_bad_days, 2) AS winter_pct_bad_days,
        
        -- Trend analysis
        t.first_year,
        t.last_year,
        ROUND(t.early_aqi, 1) AS early_aqi,
        ROUND(t.recent_aqi, 1) AS recent_aqi,
        ROUND(t.recent_aqi - t.early_aqi, 1) AS aqi_change,
        CASE 
            WHEN t.recent_aqi - t.early_aqi > 20 THEN 'Worsening'
            WHEN t.recent_aqi - t.early_aqi < -20 THEN 'Improving'
            ELSE 'Stable'
        END AS trend_direction,
        
        -- Market size metrics
        p.state_population,
        ROUND(p.state_population / 1000000.0, 2) AS population_millions,
        
        -- Wealth/Income proxies
        v.total_vehicles,
        ROUND(v.total_vehicles * 1000.0 / NULLIF(p.state_population, 0), 2) AS vehicles_per_1000_people,
        ROUND(v.pct_ev_vehicles, 2) AS pct_ev_vehicles,
        ROUND(v.pct_motor_cars, 2) AS pct_motor_cars,
        
        -- Health burden
        COALESCE(d.total_respiratory_cases, 0) AS respiratory_cases,
        COALESCE(d.total_respiratory_deaths, 0) AS respiratory_deaths,
        ROUND(COALESCE(d.case_fatality_rate, 0), 2) AS case_fatality_rate,
        
        -- Data quality
        a.total_monitoring_days,
        CASE 
            WHEN a.total_monitoring_days >= 1000 THEN 'High'
            WHEN a.total_monitoring_days >= 500 THEN 'Medium'
            ELSE 'Low'
        END AS data_reliability
        
    FROM city_aqi_metrics a
    LEFT JOIN city_trends t ON a.area = t.area AND a.state = t.state
    LEFT JOIN state_population p ON a.state = p.state
    LEFT JOIN state_vehicle_metrics v ON a.state = v.state
    LEFT JOIN state_disease_burden d ON a.state = d.state
),

-- Step 7: Calculate composite scores
scored_dataset AS (
    SELECT
        *,
        
        -- AQI Severity Score (0-100)
        -- Weighted: avg AQI (40%) + % bad days (40%) + volatility (20%)
        ROUND(
            (LEAST(avg_aqi / 3.0, 100) * 0.4) +  -- Max at 300 AQI
            (pct_bad_days * 0.4) +
            (LEAST(aqi_volatility / 1.0, 100) * 0.2)  -- Max at 100 stddev
        , 1) AS aqi_severity_score,
        
        -- Market Attractiveness Score (0-100)
        -- Based on population and wealth proxies
        ROUND(
            (LEAST(population_millions / 50.0 * 100, 100) * 0.5) +  -- Max at 50M
            (LEAST(vehicles_per_1000_people / 500.0 * 100, 100) * 0.3) +  -- Max at 500 per 1000
            (LEAST(pct_motor_cars / 50.0 * 100, 100) * 0.2)  -- Max at 50%
        , 1) AS market_attractiveness_score,
        
        -- Health Impact Score (0-100)
        -- Respiratory burden relative to population
        ROUND(
            LEAST(
                (respiratory_cases * 100000.0 / NULLIF(state_population, 0)) * 10,  -- Cases per 100k
                100
            )
        , 1) AS health_impact_score,
        
        -- Urgency Score (0-100)
        -- Trend worsening + winter severity
        ROUND(
            (CASE 
                WHEN aqi_change > 50 THEN 100
                WHEN aqi_change > 30 THEN 80
                WHEN aqi_change > 10 THEN 60
                WHEN aqi_change > 0 THEN 40
                ELSE 20
            END * 0.5) +
            (winter_pct_bad_days * 0.5)
        , 1) AS urgency_score
        
    FROM master_dataset
)

-- Step 8: Calculate final composite priority score and tier
SELECT
    *,
    
    -- COMPOSITE PRIORITY SCORE (weighted average)
    ROUND(
        (aqi_severity_score * 0.35) +        -- Air quality is most important
        (market_attractiveness_score * 0.25) +  -- Market size matters
        (urgency_score * 0.25) +              -- Worsening trends create urgency
        (health_impact_score * 0.15)          -- Health burden validates need
    , 1) AS composite_priority_score,
    
    -- Priority Tier (for filtering in dashboard)
    CASE 
        WHEN (
            (aqi_severity_score * 0.35) +
            (market_attractiveness_score * 0.25) +
            (urgency_score * 0.25) +
            (health_impact_score * 0.15)
        ) >= 70 THEN 'Tier 1 - Launch Cities'
        WHEN (
            (aqi_severity_score * 0.35) +
            (market_attractiveness_score * 0.25) +
            (urgency_score * 0.25) +
            (health_impact_score * 0.15)
        ) >= 50 THEN 'Tier 2 - Early Expansion'
        WHEN (
            (aqi_severity_score * 0.35) +
            (market_attractiveness_score * 0.25) +
            (urgency_score * 0.25) +
            (health_impact_score * 0.15)
        ) >= 30 THEN 'Tier 3 - Future Opportunity'
        ELSE 'Tier 4 - Low Priority'
    END AS priority_tier,
    
    -- Estimated annual health cost impact (in rupees)
    -- Assumptions: 
    -- - ₹10,000 avg treatment cost per respiratory case
    -- - Scale by AQI severity
    ROUND(
        respiratory_cases * 10000 * (avg_aqi / 150.0)
    ) AS estimated_annual_health_cost,
    
    -- Market size estimate (addressable households)
    -- Assumptions:
    -- - Avg household size: 4.5 people
    -- - Only cities with >40% bad days have real demand
    -- - 25% of households can afford (₹15-25K product)
    CASE 
        WHEN pct_bad_days >= 40 THEN 
            ROUND(state_population / 4.5 * 0.25)
        WHEN pct_bad_days >= 20 THEN 
            ROUND(state_population / 4.5 * 0.10)
        ELSE 0
    END AS addressable_households

FROM scored_dataset
WHERE total_monitoring_days >= 365  -- At least 1 year of data
ORDER BY composite_priority_score DESC

""").fetch_df().to_csv('master_dashboard_dataset.csv', index=False)